# 01b — Raw HOG Features Per Image Type

Extracts HOG features **separately for each of the 5 MRI image types** and
concatenates them into a single wide row per patient — no averaging, no PCA.

| Image type | Pattern | HOG columns |
|------------|---------|-------------|
| Coronal | `*_t88_gfc_cor_*.gif` | `cor_HOG_0` … `cor_HOG_8099` |
| Sagittal atlas | `*_t88_gfc_sag_*.gif` | `gfc_sag_HOG_0` … `gfc_sag_HOG_8099` |
| Transverse atlas | `*_t88_gfc_tra_*.gif` | `gfc_tra_HOG_0` … `gfc_tra_HOG_8099` |
| Masked transverse | `*_t88_masked_gfc_tra_*.gif` | `masked_tra_HOG_0` … `masked_tra_HOG_8099` |
| Native sagittal | `*_sbj_111_sag_*.gif` | `sbj_sag_HOG_0` … `sbj_sag_HOG_8099` |

**Output:** `raw_hog_features_per_image.csv` — 416 rows × 40,501 columns
(`patient_id` + 5 × 8,100 HOG features)

In [15]:
import glob
import os

import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import hog
from skimage.transform import resize

## Step 1 — Parameters

HOG parameters are identical to all other notebooks.
Each image produces an 8,100-element feature vector
(15×15 blocks × 2×2 cells × 9 orientations).
Five image types × 8,100 = **40,500 features per patient**.

In [16]:
DATA_DIR   = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data"
OUTPUT_CSV = os.path.join(DATA_DIR, "notebooks", "raw_hog_features_per_image.csv")

TARGET_SIZE      = (128, 128)
ORIENTATIONS     = 9
PIXELS_PER_CELL  = (8, 8)
CELLS_PER_BLOCK  = (2, 2)
HOG_LEN          = 8_100   # features per image

IMAGE_TYPES = {
    "cor":        "*_t88_gfc_cor_*.gif",
    "gfc_sag":    "*_t88_gfc_sag_*.gif",
    "gfc_tra":    "*_t88_gfc_tra_*.gif",
    "masked_tra": "*_t88_masked_gfc_tra_*.gif",
    "sbj_sag":    "*_sbj_111_sag_*.gif",
}

print(f"Image types      : {len(IMAGE_TYPES)}")
print(f"HOG per image    : {HOG_LEN:,}")
print(f"Total HOG cols   : {len(IMAGE_TYPES) * HOG_LEN:,}")
print(f"Expected CSV cols: {1 + len(IMAGE_TYPES) * HOG_LEN:,}  (patient_id + HOG)")

Image types      : 5
HOG per image    : 8,100
Total HOG cols   : 40,500
Expected CSV cols: 40,501  (patient_id + HOG)


## Step 2 — Helper functions

In [17]:
def load_image(path):
    img = Image.open(path).convert("L")
    arr = np.array(img, dtype=np.float32) / 255.0

    # Pad to square using the larger dimension (preserves aspect ratio)
    h, w = arr.shape
    side  = max(h, w)
    pad   = np.zeros((side, side), dtype=np.float32)
    row_off = (side - h) // 2
    col_off = (side - w) // 2
    pad[row_off:row_off + h, col_off:col_off + w] = arr

    return resize(pad, TARGET_SIZE, anti_aliasing=True)


def extract_hog(arr):
    features, _ = hog(
        arr,
        orientations=ORIENTATIONS,
        pixels_per_cell=PIXELS_PER_CELL,
        cells_per_block=CELLS_PER_BLOCK,
        visualize=True,
        feature_vector=True,
    )
    return features   # shape: (8100,)

## Step 3 — Discover patient folders

In [18]:
patient_dirs = sorted(
    d for d in glob.glob(os.path.join(DATA_DIR, "OAS1_*"))
    if os.path.isdir(d)
)
patient_ids = [os.path.basename(d) for d in patient_dirs]
N = len(patient_ids)
print(f"Found {N} patient folders")

Found 416 patient folders


## Step 4 — Extract HOG per image type

For each patient, extract HOG from each of the 5 image types in order
and concatenate into a single 40,500-element row.
If a patient is missing one image type, that block is filled with zeros
and a warning is printed.

In [19]:
# Pre-allocate output matrix: (N, 5 × 8100)
n_total = len(IMAGE_TYPES) * HOG_LEN
hog_matrix = np.zeros((N, n_total), dtype=np.float32)

for p_idx, (pid, p_dir) in enumerate(zip(patient_ids, patient_dirs)):
    row_parts = []

    for type_key, pattern in IMAGE_TYPES.items():
        matches = glob.glob(os.path.join(p_dir, pattern))
        if matches:
            row_parts.append(extract_hog(load_image(matches[0])))
        else:
            print(f"  WARNING [{type_key}]: no match in {pid} — filling with zeros")
            row_parts.append(np.zeros(HOG_LEN, dtype=np.float32))

    hog_matrix[p_idx] = np.concatenate(row_parts)

    if (p_idx + 1) % 50 == 0 or (p_idx + 1) == N:
        print(f"  Processed {p_idx + 1}/{N} patients")

print(f"\nHOG matrix shape : {hog_matrix.shape}")
print(f"Expected shape   : ({N}, {n_total})")

  Processed 50/416 patients
  Processed 100/416 patients
  Processed 150/416 patients
  Processed 200/416 patients
  Processed 250/416 patients
  Processed 300/416 patients
  Processed 350/416 patients
  Processed 400/416 patients
  Processed 416/416 patients

HOG matrix shape : (416, 40500)
Expected shape   : (416, 40500)


## Step 5 — Build DataFrame and save

Column names follow the pattern `<image_type>_HOG_<index>`,
e.g. `cor_HOG_0`, `gfc_sag_HOG_0`, …, `sbj_sag_HOG_8099`.

In [20]:
col_names = [
    f"{type_key}_HOG_{i}"
    for type_key in IMAGE_TYPES
    for i in range(HOG_LEN)
]

df = pd.DataFrame(hog_matrix, columns=col_names)
df.insert(0, "patient_id", patient_ids)

df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved  : {OUTPUT_CSV}")
print(f"Shape  : {df.shape}")
print(f"  rows  = {df.shape[0]}  (patients)")
print(f"  cols  = {df.shape[1]}  (patient_id + {df.shape[1]-1:,} HOG features)")

Saved  : C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\raw_hog_features_per_image.csv
Shape  : (416, 40501)
  rows  = 416  (patients)
  cols  = 40501  (patient_id + 40,500 HOG features)


## Step 6 — Preview

In [21]:
print(f"First 3 rows, first 10 columns:")
df.iloc[:3, :10]

First 3 rows, first 10 columns:


,patient_id,cor_HOG_0,cor_HOG_1,cor_HOG_2,cor_HOG_3,cor_HOG_4,cor_HOG_5,cor_HOG_6,cor_HOG_7,cor_HOG_8
0,OAS1_0001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,OAS1_0002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,OAS1_0003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
